# 공개용 작업 증빙 사본
[확실] 기존 파일의 코드·설명을 복사하고 출력·실행 번호·첨부 이미지를 제거했습니다. 개인 경로는 `private_inputs`로 바꿨습니다. 원자료 및 상세 결과는 포함하지 않습니다.
[확실] 이 사본을 새로 전체 실행한 기록은 아닙니다. 과거 실행 범위는 `evidence/verification.json`과 `docs/VALIDATION.md`를 확인하세요.


# 월별·연령별 활동 저조 우선 점검 공간
VS Code → Python 3.11 이상 커널 → 설치 셀 → 모두 실행.

[확실] 원본을 수정하지 않습니다. 결과는 main work/monthly_grid_results/실행시각 폴더에 저장합니다.
[추정] 50m 격자는 개인이 아닙니다. 결과는 고립 진단이 아니라 유사 환경 대비 활동 저조 공간입니다. 하위 비율 24.4/26.5/29.5/37.2/39.4%는 사용자 지정 선별 규모 가정이며 19세 차이는 보정하지 않습니다.
[확실] 새 전연령 인구는 2025년 12월 단면입니다. 기존 월별 총인구·20대 인구와 구분합니다.
[추정] 전연령 인구는 12월 연령비중을 고정 환경 변수로 6개월 모두 사용합니다. 7~11월 실제 연령별 인구로 간주하지 않습니다.
[불확실] 건물 2026-10과 유동 2025년 사이 시점 차이가 있습니다. 사용승인일이 분석월 이후인 건물은 제외하지만 멸실·용도변경은 복원할 수 없습니다.
[추정] UTM-K(EPSG:5179) 좌표를 중심으로 ±25m 사각형을 재구성합니다.

In [ ]:
%pip install -q pandas numpy geopandas pyogrio shapely scipy

## 1. 설정
[추정] 같은 지역·월·연령에서 건물 구성과 동 인구 환경이 가까운 50개 격자를 비교합니다. 자기 자신은 제외합니다.
점수 = log1p(관측 유동) - log1p(비교집단 유동 중앙값). 작을수록 활동 저조입니다.
[추정] 분석 대상은 주거 연면적>0이고 건물 용도·연면적 확인 면적 비중≥80%인 격자입니다. 최소 비교집단은 10개입니다.
[추정] 동점은 GRID_ID 오름차순으로 끊어 floor(N×비율)개 선정하며 경계 동점과 실제 비율을 기록합니다.

In [ ]:
from pathlib import Path
from datetime import datetime
import json,re,math,zipfile,hashlib,xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
from shapely.geometry import box
from IPython.display import display
ROOT=Path(r'./private_inputs/main_work')
OUT=ROOT/'monthly_grid_results'/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT.mkdir(parents=True,exist_ok=True)
MONTHS=[f'2025{m:02}' for m in range(7,13)]
RATES={'20G':.244,'30G':.265,'40G':.295,'50G':.372,'60GU':.394}
K=50
MIN_PEERS=10
MIN_KNOWN_SHARE=.80
FILES=[p for p in ROOT.rglob('*') if p.is_file() and 'monthly_grid_results' not in p.parts and not p.name.startswith('~$')]
AUDIT=[]
KEY=['STD_YM','BLOCK_CD','X_COORD','Y_COORD']
def save(d,n):d.to_csv(OUT/n,index=False,encoding='utf-8-sig')
def find(pattern):
    f=[p for p in FILES if p.match(pattern)]
    if len(f)!=1:raise ValueError(f'{pattern}: 파일 {len(f)}개. 중복/누락 확인: {f}')
    return f[0]
print('결과 저장 위치:',OUT)

## 2. 읽기 전용 XLSX 파서·월별 인구
[확실] 설치가 덜 된 환경에서도 엑셀 원본을 읽을 수 있도록 표준 XLSX XML 파서를 사용합니다. 현재 확인한 첫 시트 구조에 맞추었고 구조가 바뀌면 중단합니다.

In [ ]:
def xlsx_rows(path,sheet=1):
    ns={'m':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    with zipfile.ZipFile(path) as z:
        ss=[]
        if 'xl/sharedStrings.xml' in z.namelist():ss=[''.join(t.itertext()) for t in ET.fromstring(z.read('xl/sharedStrings.xml')).findall('m:si',ns)]
        rows=[]
        for row in ET.fromstring(z.read(f'xl/worksheets/sheet{sheet}.xml')).findall('.//m:row',ns):
            vals={}
            for c in row:
                v=c.find('m:v',ns)
                vals[re.sub('[0-9]','',c.attrib['r'])]=ss[int(v.text)] if c.attrib.get('t')=='s' and v is not None else (v.text if v is not None else ''.join(c.itertext()))
            rows.append(vals)
        return rows
def col(i):
    s=''
    while i:i,r=divmod(i-1,26);s=chr(65+r)+s
    return s
def num(v):return float(str(v).replace(',','').strip())
monthly=[]
for region in ['강남','춘천']:
    fs=[p for p in FILES if p.suffix=='.xlsx' and '연령별인구현황' in p.name and region in p.name]
    if len(fs)!=1:raise ValueError(f'{region} 월별인구 파일 누락/중복')
    rows=xlsx_rows(fs[0]);header=next(r for r in rows if r.get('A')=='행정기관코드')
    for j,month in enumerate(MONTHS):
        c=3+9*j
        if header.get(col(c+2))!='20~29세':raise ValueError('월별인구 구조 변경')
        for r in rows:
            ac=r.get('A','')
            if not re.fullmatch(r'\d{10}',ac) or ac.endswith('00000'):continue
            total=num(r[col(c)]);young=num(r[col(c+2)])
            monthly.append(dict(REGION='강남구' if region=='강남' else '춘천시',DONG=r['B'].strip().split()[-1],STD_YM=month,MOIS_CODE=ac,MONTH_TOTAL=total,MONTH_20=young,MONTH_20_SHARE=young/total if total else np.nan))
monthly=pd.DataFrame(monthly)
assert not monthly.duplicated(['REGION','DONG','STD_YM']).any()
save(monthly,'00_monthly_population.csv')

## 3. 새 연령별 인구 집계
[확실] 남·여가 아닌 '계' 행만 읽고 1세별 값을 20·30·40·50·60세 이상으로 합칩니다. 시군구 합계 열은 제외하고 동별 합계와 0세~최고령 합산값을 검산합니다.
[확실] 강남의 12월 전체 인구와 기존 월간 '거주자수'는 포괄 범위가 달라 값이 다릅니다. 두 자료의 분모를 섞지 않고 새 파일 안의 총인구로 새 연령비중을 계산합니다.

In [ ]:
snap=[]
for region,pattern in [('강남구','*강남*기관별*.xlsx'),('춘천시','*춘천*붙임2*.xlsx')]:
    p=find(pattern);rows=xlsx_rows(p)
    h=next(i for i,r in enumerate(rows) if r.get('B')=='연령')
    names=rows[h];types=rows[h+1]
    columns=[c for c,name in names.items() if types.get(c)=='인구수' and name not in ['강남구','춘천시','']]
    totalrow=next(r for r in rows[h+2:] if r.get('C')=='계' and '합' in r.get('B',''))
    annual=[r for r in rows[h+2:] if r.get('C')=='계' and re.fullmatch(r'\d+세(?:\s*이상)?',r.get('B','').strip())]
    if len(annual)<100:raise ValueError('1세별 인구 표를 확인하세요')
    for c in columns:
        total=num(totalrow[c]);age_values={int(re.match(r'\d+',r['B']).group()):num(r[c]) for r in annual}
        if not np.isclose(sum(age_values.values()),total):raise ValueError(f'{region}/{names[c]} 연령합계 불일치')
        for a in RATES:
            lo=int(a[:2]);hi=200 if a=='60GU' else lo+10
            count=sum(v for age,v in age_values.items() if lo<=age<hi)
            snap.append(dict(REGION=region,DONG=names[c].strip(),AGE=a,SNAPSHOT_MONTH='202512',SNAPSHOT_TOTAL=total,SNAPSHOT_AGE_POP=count,SNAPSHOT_AGE_SHARE=count/total if total else np.nan,SNAPSHOT_SOURCE=p.name))
snapshot=pd.DataFrame(snap)
assert not snapshot.duplicated(['REGION','DONG','AGE']).any()
save(snapshot,'01_age_population_snapshot_202512.csv')
display(snapshot.head())

## 4. SK 데이터와 행정경계
[확실] 구분자 |, 병합키는 월+BLOCK_CD+X+Y. 완전 중복만 제거합니다. 같은 키의 상충 값은 중단합니다.
[확실] 경계 SHP는 UTF-8이고 SGIS ADM_CD는 8자리, 인구의 행정기관코드는 10자리입니다. 서로 그대로 연결하지 않고 지역+동명을 검증해 연결합니다.
[추정] 경계선 다중 일치는 검토 대상으로 제외합니다. 지역 밖 격자와 좌표가 같고 BLOCK_CD가 다른 격자도 기록합니다.

In [ ]:
def read_flow(kind,month):
    p=find(f'flow_{kind}_pop_{month}.csv')
    d=pd.read_csv(p,sep='|',dtype={'STD_YM':str,'BLOCK_CD':str});d.columns=d.columns.str.strip()
    if not set(KEY)<=set(d):raise ValueError('CSV 필수 컬럼 누락')
    n=len(d);d=d.drop_duplicates().copy()
    AUDIT.append(dict(kind=kind,month=month,path=str(p),rows_raw=n,duplicates_removed=n-len(d)))
    if d.duplicated(KEY).any():
        save(d[d.duplicated(KEY,False)],f'ERROR_conflict_{kind}_{month}.csv');raise ValueError('동일 키 값 충돌')
    if set(d.STD_YM)!={month}:raise ValueError('월 불일치')
    for c in d.columns.difference(['STD_YM','BLOCK_CD']):d[c]=pd.to_numeric(d[c],errors='coerce')
    if not (d.X_COORD.between(700000,1400000)&d.Y_COORD.between(1400000,2200000)).all():raise ValueError('UTM-K 좌표 범위 확인')
    return d
ages=pd.concat([read_flow('age',m) for m in MONTHS],ignore_index=True)
for a in RATES:
    cs=[f'MAN_FLOW_POP_CNT_{a}',f'WMAN_FLOW_POP_CNT_{a}']
    ages[a]=ages[cs].sum(axis=1,min_count=2)
    ages.loc[(ages[cs]<0).any(axis=1),a]=np.nan
boundaries=[]
for cd,region in [('11230','강남구'),('32010','춘천시')]:
    p=find(f'bnd_dong_{cd}_2025_2Q.shp')
    b=pyogrio.read_dataframe(p)
    if b.crs is None:raise ValueError('경계 CRS 없음')
    b=b.to_crs(5179).rename(columns={'ADM_NM':'DONG'})
    b['REGION']=region;boundaries.append(b[['BASE_DATE','ADM_CD','DONG','REGION','geometry']])
bnd=gpd.GeoDataFrame(pd.concat(boundaries,ignore_index=True),geometry='geometry',crs=5179)
assert not bnd.duplicated(['REGION','DONG']).any()
s=ages[['BLOCK_CD','X_COORD','Y_COORD']].drop_duplicates().copy()
s['GRID_ID']=s.BLOCK_CD+'_'+s.X_COORD.map(lambda v:format(v,'.8f'))+'_'+s.Y_COORD.map(lambda v:format(v,'.8f'))
points=gpd.GeoDataFrame(s,geometry=gpd.points_from_xy(s.X_COORD,s.Y_COORD),crs=5179)
j=gpd.sjoin(points,bnd,how='left',predicate='intersects')
multiple=j.GRID_ID[j.GRID_ID.duplicated(False)]
j['SPATIAL_STATUS']=np.where(j.GRID_ID.isin(multiple),'boundary_multiple',np.where(j.ADM_CD.isna(),'outside','ok'))
save(pd.DataFrame(j.drop(columns='geometry')),'02_spatial_audit.csv')
v=j[j.SPATIAL_STATUS=='ok'].drop(columns=['geometry','index_right','SPATIAL_STATUS']).copy()
ages=ages.merge(v,on=['BLOCK_CD','X_COORD','Y_COORD'],validate='many_to_one')
# 같은 월·물리 좌표의 여러 코드는 중복 계수하지 않고 중단
dup=ages.duplicated(['STD_YM','X_COORD','Y_COORD'],False)
if dup.any():
    save(ages[dup],'ERROR_same_location_codes.csv');raise ValueError('같은 월·좌표에 복수 코드: ERROR_same_location_codes.csv 확인')
ages=ages.merge(monthly,on=['REGION','DONG','STD_YM'],how='left',validate='many_to_one')
missing=set(zip(v.REGION,v.DONG))-set(zip(snapshot.REGION,snapshot.DONG))
if missing or ages.MONTH_TOTAL.isna().any():raise ValueError(f'인구/경계 동명 불일치: {missing}')
grids=gpd.GeoDataFrame(v,geometry=[box(x-25,y-25,x+25,y+25) for x,y in zip(v.X_COORD,v.Y_COORD)],crs=5179)
display(ages.groupby(['REGION','STD_YM']).size())

## 5. 건물-격자 결합
[확실] A9 용도명, A13 사용승인일, A14 연면적입니다. 대용량 시도 건물에서 시군구 조건으로 필요한 속성만 읽습니다.
[추정] 건물 주용도로 분류하고 겹치는 외곽선 면적 비율로 연면적을 배분합니다. 복합용도의 실제 세부 면적이 아닙니다. 결측/0 연면적을 임의 보충하지 않습니다.

In [ ]:
bs=[]
for prefix,region,sgg in [('11','강남구','11680'),('51','춘천시','51110')]:
    p=find(f'AL_D010_{prefix}_*.shp')
    info=pyogrio.read_info(p)
    bounds=tuple(bnd[bnd.REGION==region].to_crs(info['crs']).total_bounds)
    b=pyogrio.read_dataframe(p,bbox=bounds,columns=['A1','A4','A9','A13','A14','A26'],encoding='cp949')
    b=b[b.A4.fillna('').str.contains(region)].copy()
    if b.empty or b.crs is None:raise ValueError('건물 또는 좌표계 없음')
    b=b.to_crs(5179);b.geometry=b.geometry.make_valid()
    b=b[b.geometry.notna() & ~b.geometry.is_empty].copy();b['REGION']=region;bs.append(b)
buildings=gpd.GeoDataFrame(pd.concat(bs,ignore_index=True),geometry='geometry',crs=5179)
buildings['BID']=np.arange(len(buildings));buildings['FOOTPRINT']=buildings.area
buildings=buildings[buildings.FOOTPRINT>0].copy()
buildings['GFA']=pd.to_numeric(buildings.A14,errors='coerce')
buildings['APPROVED']=pd.to_datetime(buildings.A13,errors='coerce')
u=buildings.A9.fillna('').astype(str)
buildings['CLASS']=np.select([u.str.contains('단독주택'),u.str.contains('공동주택|아파트|연립주택|다세대주택'),u.str.contains('근린생활|판매|업무|숙박|위락')],['detached','multi','commerce'],default='other')
buildings['KNOWN']=(buildings.GFA>0)&u.ne('')
q=gpd.sjoin(grids[['GRID_ID','REGION','geometry']],buildings[['BID','geometry']],how='inner',predicate='intersects').reset_index(drop=True)
bg=buildings.set_index('BID')
q['OVERLAP']=q.geometry.intersection(gpd.GeoSeries(bg.loc[q.BID].geometry.to_numpy(),index=q.index,crs=5179)).area
q=q[q.OVERLAP>0].drop(columns=['geometry','index_right']).merge(pd.DataFrame(buildings.drop(columns='geometry')),on='BID',suffixes=('','_B'))
q=q[q.REGION==q.REGION_B].copy();q['ALLOC']=q.GFA*q.OVERLAP/q.FOOTPRINT
save(pd.DataFrame(buildings.drop(columns='geometry')),'03_building_inventory.csv')
def building_features(month):
    end=pd.Timestamp(month+'01')+pd.offsets.MonthEnd(0)
    p=q[q.APPROVED.isna()|(q.APPROVED<=end)].copy()
    p['KNOWN_AREA']=p.OVERLAP.where(p.KNOWN,0);p['UNKNOWN_DATE_AREA']=p.OVERLAP.where(p.APPROVED.isna(),0)
    for c in ['detached','multi','commerce','other']:p[c]=p.ALLOC.where(p.KNOWN&(p.CLASS==c),0)
    f=p.groupby('GRID_ID').agg(OVERLAP_AREA=('OVERLAP','sum'),KNOWN_AREA=('KNOWN_AREA','sum'),UNKNOWN_DATE_AREA=('UNKNOWN_DATE_AREA','sum'),**{c:(c,'sum') for c in ['detached','multi','commerce','other']}).reset_index()
    f['KNOWN_SHARE']=f.KNOWN_AREA/f.OVERLAP_AREA;f['DATE_UNKNOWN_SHARE']=f.UNKNOWN_DATE_AREA/f.OVERLAP_AREA
    f['RES_GFA']=f.detached+f.multi;f['TOTAL_GFA']=f[['detached','multi','commerce','other']].sum(axis=1)
    f['MULTI_SHARE']=f.multi/f.RES_GFA.replace(0,np.nan);f['COMMERCE_SHARE']=f.commerce/f.TOTAL_GFA.replace(0,np.nan);f['RES_SHARE']=f.RES_GFA/f.TOTAL_GFA.replace(0,np.nan)
    return f
print('대상 건물',len(buildings),'교차',len(q))

## 6. 월별 점수·선정
[추정] 표준화한 주거 규모·전체 규모·공동주택 비중·상업업무 비중·주거 비중·월 총인구·월 20대 비중·12월 해당 연령 비중을 같은 가중치로 비교합니다. 동 인구는 환경 변수이며 격자 거주인구로 배분하지 않습니다.
[확실] 시간·요일 자료는 전체 연령 공간 설명용입니다. 연령×시간·요일 값으로 해석하지 않습니다.

In [ ]:
def context(month):
    t=read_flow('time',month);w=read_flow('wkdy',month)
    for c,hs in [('DAY_ALL',range(9,18)),('EVENING_ALL',range(18,22))]:
        cs=[f'TMST_{h:02}' for h in hs];t[c]=t[cs].mean(axis=1).where(t[cs].notna().all(axis=1))
    for c,ds in [('WEEKDAY_ALL',['MON','TUS','WED','THU','FRI']),('WEEKEND_ALL',['SAT','SUN'])]:
        cs=['FLOW_POP_CNT_'+d for d in ds];w[c]=w[cs].mean(axis=1).where(w[cs].notna().all(axis=1))
    w['WEEKEND_RATIO_ALL']=w.WEEKEND_ALL/w.WEEKDAY_ALL.replace(0,np.nan)
    return t[KEY+['DAY_ALL','EVENING_ALL']].merge(w[KEY+['WEEKDAY_ALL','WEEKEND_ALL','WEEKEND_RATIO_ALL']],on=KEY,how='outer',validate='one_to_one')
def neighbor_arrays(x,k):
    # scipy가 없는 검증 환경에서도 실행 가능한 NumPy 대체 경로
    try:
        from scipy.spatial import cKDTree
        dist,nn=cKDTree(x).query(x,k=min(k+1,len(x)))
        peers=[];mean_dist=[]
        for i in range(len(x)):
            mask=nn[i]!=i;peers.append(nn[i][mask][:k]);mean_dist.append(dist[i][mask][:k].mean())
        return peers,np.array(mean_dist)
    except ImportError:
        peers=[];mean_dist=[]
        for i in range(len(x)):
            dd=np.sum((x-x[i])**2,axis=1);dd[i]=np.inf
            js=np.argpartition(dd,k-1)[:k];peers.append(js);mean_dist.append(np.sqrt(dd[js]).mean())
        return peers,np.array(mean_dist)
results=[];summaries=[]
for month in MONTHS:
    print('처리:',month,flush=True)
    d=ages[ages.STD_YM==month].merge(building_features(month),on='GRID_ID',how='left',validate='one_to_one').merge(context(month),on=KEY,how='left',validate='one_to_one')
    d['EXCLUSION']=np.select([d.OVERLAP_AREA.isna(),d.KNOWN_SHARE.lt(MIN_KNOWN_SHARE),d.RES_GFA.le(0)],['no_building','building_attributes_incomplete','no_residential_gfa'],default='')
    save(d[d.EXCLUSION!=''],f'excluded_{month}.csv')
    for region,part in d[d.EXCLUSION==''].groupby('REGION'):
        for age,rate in RATES.items():
            a=part.merge(snapshot[snapshot.AGE==age],on=['REGION','DONG'],how='left',validate='many_to_one')
            a['FLOW']=a[age]
            bad=a.FLOW.isna()|a.SNAPSHOT_AGE_SHARE.isna()
            ex=a[bad].copy();ex['EXCLUSION']='missing_flow_or_age_environment';save(ex,f'excluded_{region}_{month}_{age}.csv')
            a=a[~bad].copy().sort_values('GRID_ID').reset_index(drop=True)
            if len(a)<=MIN_PEERS:
                a['EXCLUSION']='insufficient_peers';save(a,f'excluded_peers_{region}_{month}_{age}.csv')
                summaries.append(dict(REGION=region,STD_YM=month,AGE=age,eligible=len(a),selected=0,target_rate=rate,status='insufficient_peers'));continue
            f=pd.DataFrame(dict(log_res=np.log1p(a.RES_GFA),log_total=np.log1p(a.TOTAL_GFA),multi=a.MULTI_SHARE,commerce=a.COMMERCE_SHARE,res_share=a.RES_SHARE,log_dong_pop=np.log1p(a.MONTH_TOTAL),month_20_share=a.MONTH_20_SHARE,age_share_dec=a.SNAPSHOT_AGE_SHARE))
            assert np.isfinite(f.to_numpy()).all()
            x=((f-f.mean())/f.std(ddof=0).replace(0,1)).to_numpy()
            peers,dist=neighbor_arrays(x,min(K,len(a)-1));y=a.FLOW.to_numpy()
            a['PEER_MEDIAN']=[np.median(y[js]) for js in peers];a['PEER_COUNT']=[len(js) for js in peers];a['PEER_DISTANCE']=dist
            a['PEER_DISTANCE_REVIEW']=a.PEER_DISTANCE>a.PEER_DISTANCE.quantile(.9)
            a['ACTIVITY_RESIDUAL']=np.log1p(a.FLOW)-np.log1p(a.PEER_MEDIAN)
            a=a.sort_values(['ACTIVITY_RESIDUAL','GRID_ID'],kind='stable').reset_index(drop=True)
            n=len(a);count=math.floor(n*rate);a['RANK']=np.arange(1,n+1);a['SELECTED']=a.RANK<=count
            a['TARGET_RATE']=rate;a['ACTUAL_RATE']=count/n
            cutoff=a.iloc[count-1].ACTIVITY_RESIDUAL if count else np.nan;a['CUTOFF_TIE']=a.ACTIVITY_RESIDUAL.eq(cutoff)
            summaries.append(dict(REGION=region,STD_YM=month,AGE=age,eligible=n,selected=count,target_rate=rate,actual_rate=count/n,cutoff_score=cutoff,cutoff_tied_total=int(a.CUTOFF_TIE.sum()),status='ok'))
            save(a,f'all_{region}_{month}_{age}.csv');save(a[a.SELECTED],f'selected_{region}_{month}_{age}.csv');results.append(a)
if not results:raise ValueError('분석 가능 결과 없음')
result=pd.concat(results,ignore_index=True);summary=pd.DataFrame(summaries)
save(result,'04_all_scores.csv');save(result[result.SELECTED],'05_selected_spaces.csv');save(summary,'06_selection_summary.csv')
dong=result.groupby(['REGION','STD_YM','AGE','ADM_CD','DONG']).agg(eligible=('GRID_ID','size'),selected=('SELECTED','sum')).reset_index();dong['selected_share']=dong.selected/dong.eligible
save(dong,'07_dong_summary.csv');save(pd.DataFrame(AUDIT),'08_input_audit.csv')
display(summary)

## 7. 지도·GIS·결과 확인
[확실] 지역·월별 지도에서 연령을 선택할 수 있습니다. 지도 배경과 Leaflet 라이브러리는 인터넷이 필요하고 CSV·GeoJSON은 오프라인 사용 가능합니다.

In [ ]:
selected=result[result.SELECTED].copy()
for (region,month),part in selected.groupby(['REGION','STD_YM']):
    geo=gpd.GeoDataFrame(part.merge(grids[['GRID_ID','geometry']],on='GRID_ID',validate='many_to_one'),geometry='geometry',crs=5179).to_crs(4326)
    cs=['GRID_ID','AGE','DONG','FLOW','PEER_MEDIAN','ACTIVITY_RESIDUAL','RANK','SELECTED','geometry']
    gj=geo[cs].to_json()
    (OUT/f'selected_{region}_{month}.geojson').write_text(gj,encoding='utf-8')
    title=f'{region} {month} 활동 저조 점검 공간'
    html="""<!doctype html><html lang="ko"><meta charset="utf-8"><title>__TITLE__</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<style>body{margin:0;font:14px sans-serif}header{padding:12px}#map{height:90vh}</style>
<header><b>__TITLE__</b> <select id="age"><option>20G</option><option>30G</option><option>40G</option><option>50G</option><option>60GU</option></select> <span id="count"></span><br>사회적 고립 확정이 아닌 공간별 우선 점검 목록</header><div id="map"></div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script><script>
const data=__DATA__,m=L.map('map'),layer=L.geoJSON(null,{style:{color:'#2563eb',weight:1,fillOpacity:.6},onEachFeature:(f,l)=>{const p=f.properties;l.bindTooltip(p.DONG+' / '+p.AGE+' / 유동 '+p.FLOW+' / 순위 '+p.RANK)}}).addTo(m);
L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',{attribution:'© OpenStreetMap'}).addTo(m);
function draw(){const a=document.getElementById('age').value,fs=data.features.filter(f=>f.properties.AGE===a);layer.clearLayers();layer.addData({type:'FeatureCollection',features:fs});if(fs.length)m.fitBounds(layer.getBounds());document.getElementById('count').textContent=fs.length+'개 격자';}
document.getElementById('age').onchange=draw;draw();</script></html>"""
    html=html.replace('__TITLE__',title).replace('__DATA__',gj.replace('</','<\\/'))
    (OUT/f'map_{region}_{month}.html').write_text(html,encoding='utf-8')
assert not result.duplicated(['REGION','STD_YM','AGE','GRID_ID']).any()
for r in summary[summary.status=='ok'].itertuples():assert r.selected==math.floor(r.eligible*r.target_rate)
assert (result.PEER_COUNT>=MIN_PEERS).all() and result.ACTIVITY_RESIDUAL.notna().all()
manifest=dict(root=str(ROOT),output=str(OUT),months=MONTHS,rates=RATES,k=K,min_peers=MIN_PEERS,min_known_building_share=MIN_KNOWN_SHARE,building_snapshot='20261009',age_population_snapshot='202512 fixed environmental composition across all months',boundary_snapshot='20250630',sources=[dict(path=str(p),bytes=p.stat().st_size) for p in FILES if p.suffix in ['.shp','.xlsx'] or p.name.startswith('flow_')],limitations=['spatial screening, not person diagnosis','fixed prevalence-based quota is an assumption','2026 buildings vs 2025 flow','December age composition used for earlier months, retrospective use','building main use and footprint allocation approximate mixed use','50m squares reconstructed around points'])
(OUT/'09_run_metadata.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('완료:',OUT)
print('핵심 결과: 05_selected_spaces.csv / 06_selection_summary.csv / map_지역_월.html')

## 결과 파일
|파일|용도|
|---|---|
|05_selected_spaces.csv|월·지역·연령별 선정 공간과 좌표·행정동·점수|
|04_all_scores.csv|선정·미선정 전체 분석 가능 공간|
|06_selection_summary.csv|모집단 수·선정 수·실제 비율·동점|
|07_dong_summary.csv|동별 선정 격자 수·비율|
|excluded_*.csv|건물 누락·비주거·연령 누락·이웃 부족 사유|
|02_spatial_audit.csv|경계 밖·경계 다중 일치 포함 공간 연결 기록|
|map_*.html / selected_*.geojson|월별 지도 / GIS 도형|
|00·01 인구 / 03 건물 / 08·09 검증|실제 사용 입력·중복 제거·설정 기록|

[확실] 유동 수치가 낮다는 사실만으로 해당 연령 주민의 외출률·고립 여부를 알 수 없습니다.
[추정] 비교집단 선택과 선별 비율은 분석 가정입니다. 실제 사업 적용은 현장 검증이 필요합니다.
공식 고립도 참고: https://www.index.go.kr/unity/potal/indicator/IndexInfo.do?clasCd=8&idxCd=8080
건물 필드 참고: https://mods.go.kr/boardDownload.es?bid=12053&list_no=391854&seq=1